# CitaPred: Citation Prediction System

**Master's Capstone Project - American University of Beirut (AUB)**

This notebook demonstrates the CitaPred system for predicting citation impact of academic papers.

## Features
- 📊 **Classification**: Predict highly cited vs not (82.74% accuracy)
- 📈 **Regression**: Predict citation counts (R²=0.14, Spearman=0.72)
- 🎯 **Feature Analysis**: Visualize feature importance
- 📁 **Batch Processing**: Process multiple papers at once

---

## Setup and Imports

In [ ]:
# Standard libraries
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Model loading
import joblib

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

from citapred.features.extractor import FeatureExtractor

print("✅ Imports successful!")

## Load Trained Models

In [ ]:
def load_available_models():
    """Load all available trained models."""
    models_dir = Path('models')
    
    if not models_dir.exists():
        print("❌ Models directory not found!")
        print("Please train models first:")
        print("  python scripts/train_classification.py")
        print("  python scripts/train_regression.py")
        return {}
    
    models = {}
    model_files = list(models_dir.glob('*.pkl'))
    
    if not model_files:
        print("❌ No trained models found!")
        return {}
    
    for model_file in model_files:
        model_name = model_file.stem
        try:
            models[model_name] = joblib.load(model_file)
            print(f"✅ Loaded: {model_name}")
        except Exception as e:
            print(f"❌ Failed to load {model_name}: {e}")
    
    return models

# Load models
models = load_available_models()
feature_extractor = FeatureExtractor()

print(f"\n📊 Total models loaded: {len(models)}")
print(f"Available models: {list(models.keys())}")

## Helper Functions

In [ ]:
def predict_paper(paper_data, model_name='general_classification_model'):
    """
    Predict citation impact for a single paper.
    
    Parameters:
    -----------
    paper_data : dict
        Paper metadata (title, year, venue, authors, etc.)
    model_name : str
        Name of the model to use for prediction
    
    Returns:
    --------
    dict : Prediction results
    """
    if model_name not in models:
        raise ValueError(f"Model '{model_name}' not found. Available: {list(models.keys())}")
    
    # Extract features
    df = pd.DataFrame([paper_data])
    features = feature_extractor.extract_features(df)
    
    # Get model
    model = models[model_name]
    
    # Make prediction
    prediction = model.predict(features)[0]
    
    # Get probability for classification
    probability = None
    if 'classification' in model_name and hasattr(model, 'predict_proba'):
        proba = model.predict_proba(features)[0]
        probability = proba[1]  # Probability of being highly cited
    
    # Get feature importance if available
    feature_importance = None
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
        feature_names = features.columns
        feature_importance = dict(zip(feature_names, importance))
    
    return {
        'prediction': prediction,
        'probability': probability,
        'feature_importance': feature_importance,
        'features': features,
        'model_name': model_name
    }


def display_classification_result(result):
    """Display classification prediction results."""
    prediction_label = "🌟 Highly Cited" if result['prediction'] == 1 else "📄 Not Highly Cited"
    confidence = result['probability'] * 100 if result['probability'] else 50
    
    print("="*60)
    print(f"🎯 PREDICTION: {prediction_label}")
    print(f"📊 Confidence: {confidence:.1f}%")
    print(f"🤖 Model: {result['model_name']}")
    print("="*60)
    
    # Confidence gauge
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=confidence,
        title={'text': "Confidence Score (%)"},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': "darkblue"},
            'steps': [
                {'range': [0, 50], 'color': "lightgray"},
                {'range': [50, 75], 'color': "gray"},
                {'range': [75, 100], 'color': "lightgreen"}
            ],
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': 90
            }
        }
    ))
    fig.update_layout(height=400, width=600)
    fig.show()


def display_regression_result(result):
    """Display regression prediction results."""
    predicted_citations = max(0, int(result['prediction']))
    lower = int(predicted_citations * 0.8)
    upper = int(predicted_citations * 1.2)
    
    print("="*60)
    print(f"📈 PREDICTED CITATIONS: {predicted_citations:,}")
    print(f"📊 Estimated Range: {lower:,} - {upper:,}")
    print(f"🤖 Model: {result['model_name']}")
    print("="*60)
    
    # Citation estimate visualization
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=['Lower Estimate', 'Prediction', 'Upper Estimate'],
        y=[lower, predicted_citations, upper],
        marker_color=['lightblue', 'darkblue', 'lightblue'],
        text=[f"{lower:,}", f"{predicted_citations:,}", f"{upper:,}"],
        textposition='auto'
    ))
    fig.update_layout(
        title="Citation Count Prediction",
        yaxis_title="Citations",
        height=400,
        width=600
    )
    fig.show()


def plot_feature_importance(result, top_n=15):
    """Plot top N most important features."""
    if not result['feature_importance']:
        print("⚠️ Feature importance not available for this model")
        return
    
    # Sort and get top N
    importance_sorted = sorted(
        result['feature_importance'].items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]
    
    features = [f[0] for f in importance_sorted]
    importances = [f[1] for f in importance_sorted]
    
    # Create plot
    fig = px.bar(
        x=importances,
        y=features,
        orientation='h',
        labels={'x': 'Importance', 'y': 'Feature'},
        title=f'Top {top_n} Most Important Features'
    )
    fig.update_layout(
        height=500,
        width=800,
        yaxis={'categoryorder': 'total ascending'}
    )
    fig.show()


print("✅ Helper functions defined!")

---
## 📝 Single Paper Prediction

Predict citation impact for a single paper by providing its metadata.

### Example 1: High Impact Paper

In [ ]:
# Define paper metadata
paper = {
    'title': 'Attention Is All You Need',
    'abstract': 'We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.',
    'venue': 'NeurIPS',
    'year': 2017,
    'authors': [
        {'name': 'Ashish Vaswani', 'hIndex': 45},
        {'name': 'Noam Shazeer', 'hIndex': 38},
        {'name': 'Niki Parmar', 'hIndex': 25}
    ],
    'citationCount': 0,  # Placeholder
    'referenceCount': 45
}

# Make prediction
result = predict_paper(paper, model_name='general_classification_model')

# Display results
display_classification_result(result)

In [ ]:
# Show feature importance
plot_feature_importance(result, top_n=15)

### Example 2: Custom Paper (Modify Below)

In [ ]:
# ✏️ EDIT THIS: Define your own paper
custom_paper = {
    'title': 'Your Paper Title Here',
    'abstract': 'Your abstract here...',
    'venue': 'ICML',  # or your conference/journal
    'year': 2024,
    'authors': [
        {'name': 'Author 1', 'hIndex': 15},
        {'name': 'Author 2', 'hIndex': 20}
    ],
    'citationCount': 0,
    'referenceCount': 30
}

# Make prediction (choose your model)
# Options: 'general_classification_model', 'general_regression_model', 
#          'aub_classification_model', 'aub_regression_model'
result = predict_paper(custom_paper, model_name='general_classification_model')

# Display results
if 'classification' in result['model_name']:
    display_classification_result(result)
else:
    display_regression_result(result)

# Show feature importance
plot_feature_importance(result)

---
## 📁 Batch Prediction

Process multiple papers from a CSV file.

### Load Papers from CSV

In [ ]:
def batch_predict(csv_file, model_name='general_classification_model'):
    """
    Make predictions for multiple papers from a CSV file.
    
    CSV should have columns: title, year, abstract, venue, authors, max_author_hindex, etc.
    """
    # Read CSV
    df = pd.read_csv(csv_file)
    print(f"📄 Loaded {len(df):,} papers from {csv_file}")
    
    # Preview
    print("\n👀 First 3 papers:")
    print(df[['title', 'year']].head(3))
    
    predictions = []
    
    print(f"\n🔄 Processing predictions...")
    for idx, row in df.iterrows():
        try:
            # Prepare paper data
            paper_data = {
                'title': row.get('title', ''),
                'abstract': row.get('abstract', ''),
                'year': row.get('year', 2023),
                'venue': row.get('venue', ''),
                'authors': [],
                'citationCount': 0,
                'referenceCount': row.get('referenceCount', 0)
            }
            
            # Parse authors if provided
            if 'authors' in row and pd.notna(row['authors']):
                authors = str(row['authors']).split(';')
                paper_data['authors'] = [{'name': a.strip()} for a in authors]
                
                # Add h-index if available
                if 'max_author_hindex' in row and pd.notna(row['max_author_hindex']):
                    if len(paper_data['authors']) > 0:
                        paper_data['authors'][0]['hIndex'] = int(row['max_author_hindex'])
            
            result = predict_paper(paper_data, model_name)
            
            if 'classification' in model_name:
                pred_label = "Highly Cited" if result['prediction'] == 1 else "Not Highly Cited"
                confidence = result['probability'] * 100 if result['probability'] else None
                
                predictions.append({
                    'title': row.get('title', ''),
                    'year': row.get('year', ''),
                    'prediction': pred_label,
                    'confidence': f"{confidence:.1f}%" if confidence else "N/A"
                })
            else:  # Regression
                pred_citations = max(0, int(result['prediction']))
                predictions.append({
                    'title': row.get('title', ''),
                    'year': row.get('year', ''),
                    'predicted_citations': pred_citations
                })
                
        except Exception as e:
            print(f"⚠️ Error processing row {idx}: {e}")
            continue
    
    results_df = pd.DataFrame(predictions)
    print(f"\n✅ Processed {len(results_df):,} papers successfully!")
    
    return results_df


print("✅ Batch prediction function ready!")

### Example: Load and Predict

In [ ]:
# ✏️ EDIT THIS: Path to your CSV file
csv_path = 'data/processed/papers_clean.csv'  # Change to your file

# Check if file exists
if Path(csv_path).exists():
    # Run batch prediction
    results = batch_predict(csv_path, model_name='general_classification_model')
    
    # Display results
    print("\n📊 Results Preview:")
    display(results.head(10))
    
    # Save results
    output_path = 'data/predictions_results.csv'
    results.to_csv(output_path, index=False)
    print(f"\n💾 Results saved to: {output_path}")
else:
    print(f"⚠️ File not found: {csv_path}")
    print("\nPlease provide a CSV file with columns: title, year, abstract, venue, authors")

### Visualize Batch Results

In [ ]:
# For classification results
if 'prediction' in results.columns:
    # Count predictions
    counts = results['prediction'].value_counts()
    
    # Pie chart
    fig = go.Figure(data=[go.Pie(
        labels=counts.index,
        values=counts.values,
        hole=0.3
    )])
    fig.update_layout(
        title="Distribution of Predictions",
        height=500,
        width=700
    )
    fig.show()
    
    # Statistics
    total = len(results)
    highly_cited = sum(results['prediction'] == 'Highly Cited')
    print(f"\n📊 Summary Statistics:")
    print(f"   Total papers: {total:,}")
    print(f"   Highly Cited: {highly_cited:,} ({highly_cited/total*100:.1f}%)")
    print(f"   Not Highly Cited: {total-highly_cited:,} ({(total-highly_cited)/total*100:.1f}%)")

# For regression results
elif 'predicted_citations' in results.columns:
    # Distribution histogram
    fig = px.histogram(
        results,
        x='predicted_citations',
        nbins=50,
        title='Distribution of Predicted Citations'
    )
    fig.update_layout(height=500, width=800)
    fig.show()
    
    # Statistics
    print(f"\n📊 Summary Statistics:")
    print(f"   Mean: {results['predicted_citations'].mean():.0f} citations")
    print(f"   Median: {results['predicted_citations'].median():.0f} citations")
    print(f"   Max: {results['predicted_citations'].max():,.0f} citations")
    print(f"   Min: {results['predicted_citations'].min():.0f} citations")

---
## 🔍 Model Comparison

Compare predictions from different models on the same paper.

In [ ]:
def compare_models(paper_data):
    """Compare predictions from all available models."""
    
    print("🔬 Comparing Models...\n")
    print(f"Paper: {paper_data['title'][:60]}...")
    print("="*80)
    
    results = {}
    
    for model_name in models.keys():
        try:
            result = predict_paper(paper_data, model_name)
            
            if 'classification' in model_name:
                pred = "Highly Cited" if result['prediction'] == 1 else "Not Highly Cited"
                conf = f"{result['probability']*100:.1f}%" if result['probability'] else "N/A"
                print(f"\n{model_name}:")
                print(f"  Prediction: {pred}")
                print(f"  Confidence: {conf}")
            else:
                pred_cit = max(0, int(result['prediction']))
                print(f"\n{model_name}:")
                print(f"  Predicted Citations: {pred_cit:,}")
            
            results[model_name] = result
            
        except Exception as e:
            print(f"\n{model_name}: ❌ Error - {e}")
    
    print("\n" + "="*80)
    return results


# Example comparison
if len(models) > 1:
    test_paper = {
        'title': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding',
        'abstract': 'We introduce a new language representation model called BERT...',
        'venue': 'NAACL',
        'year': 2019,
        'authors': [
            {'name': 'Jacob Devlin', 'hIndex': 52},
            {'name': 'Ming-Wei Chang', 'hIndex': 48}
        ],
        'citationCount': 0,
        'referenceCount': 38
    }
    
    comparison_results = compare_models(test_paper)
else:
    print("⚠️ Need at least 2 models for comparison. Train more models first.")

---
## 📊 Model Performance Analysis

Analyze overall feature importance across the model.

In [ ]:
def analyze_model_features(model_name='general_classification_model', top_n=20):
    """Analyze and visualize model's overall feature importance."""
    
    if model_name not in models:
        print(f"❌ Model '{model_name}' not found")
        return
    
    model = models[model_name]
    
    if not hasattr(model, 'feature_importances_'):
        print("⚠️ This model doesn't support feature importance analysis")
        return
    
    # Get feature importance
    # Note: We need to extract features from a sample to get feature names
    sample_paper = {
        'title': 'Sample Paper',
        'abstract': 'Sample abstract',
        'venue': 'ICML',
        'year': 2023,
        'authors': [{'name': 'Author', 'hIndex': 10}],
        'citationCount': 0,
        'referenceCount': 20
    }
    
    df = pd.DataFrame([sample_paper])
    features = feature_extractor.extract_features(df)
    feature_names = features.columns
    
    # Create importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False).head(top_n)
    
    # Plot
    fig = px.bar(
        importance_df,
        x='importance',
        y='feature',
        orientation='h',
        title=f'Top {top_n} Most Important Features - {model_name}',
        labels={'importance': 'Importance Score', 'feature': 'Feature'}
    )
    fig.update_layout(
        height=600,
        width=900,
        yaxis={'categoryorder': 'total ascending'}
    )
    fig.show()
    
    # Print statistics
    print(f"\n📊 Feature Importance Statistics ({model_name}):")
    print(f"   Total features: {len(feature_names)}")
    print(f"\n   Top 5 Features:")
    for idx, row in importance_df.head(5).iterrows():
        print(f"   {idx+1}. {row['feature']}: {row['importance']:.4f}")


# Analyze first available model
if models:
    first_model = list(models.keys())[0]
    analyze_model_features(first_model, top_n=20)
else:
    print("⚠️ No models loaded")

---
## 🎓 Conclusion

This notebook demonstrates the CitaPred citation prediction system.

### Key Findings:
- **Classification accuracy**: 82.74% (highly cited vs not)
- **Regression performance**: R²=0.14, Spearman=0.72
- **Most important features**: Author h-index, venue prestige, publication year
- **Dataset**: 12,852 papers (6,442 General ML + 6,410 AUB)

### Next Steps:
1. Train models on AUB-specific data
2. Compare General ML vs AUB model performance
3. Analyze domain-specific citation patterns
4. Deploy for production use

---
**Master's Capstone Project - American University of Beirut (AUB)**